In [1]:
!pip install arxiv

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=21bccc9d93f2710a71200cbcde3fe37950cc4030128842beaa4fd481b06caa33
  Stored in directory: c:\users\dell\appdata\local\pip\cache\wheels\3d\4d\ef\37cdccc18d6fd7e0dd7817dcdf9146d4d6789c32a227a28134
Successfully built sgmllib3k



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# for wrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [11]:
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=200)
wiki = WikipediaQueryRun(api_wrapper=api_wrapper)

In [12]:
wiki.name

'wikipedia'

In [7]:
# for web base load langchain
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain.embeddings.ollama import OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
documents = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
vectordb = FAISS.from_documents(documents, OllamaEmbeddings(model="gemma2:2b"))
retriever =vectordb.as_retriever()
retriever

USER_AGENT environment variable not set, consider setting it to identify your requests.
C:\Users\DELL\AppData\Local\Temp\ipykernel_22240\1766451864.py:10: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  vectordb = FAISS.from_documents(documents, OllamaEmbeddings(model="gemma2:2b"))


VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000020709EDDA90>, search_kwargs={})

In [ ]:
# for anything related to langsmith
from langchain.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,"langsmith_search", "Search for information about LangSmith. For any questions about Langsmith, you must use this tool.")


In [9]:
retriever_tool.name

'langsmith_search'

In [ ]:
## Arxiv Tool # Output: Paper title + summary (max 200 characters)
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv = ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [13]:
tools = [wiki,arxiv,retriever_tool]

In [14]:
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'd:\\Project and code\\Langchain\\venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)),
 Tool(name='langsmith_search', description='Search for information about LangSmith. For any questions about Langsmith, you must use this tool.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=functools.partial(<function _get_relevant_documents at 0x000002077F9FA980>, retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_commu

In [35]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGCHAIN_API_KEY')
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(model="gemma2:2b")





C:\Users\DELL\AppData\Local\Temp\ipykernel_22240\2441737957.py:6: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gemma2:2b")


In [36]:
from langchain.prompts import PromptTemplate

ollama_prompt = PromptTemplate(
    input_variables=["input", "agent_scratchpad"],  # Use 'input' instead of 'query'
    template="""
You are an AI assistant using the Gemma 2B model. Answer the following question concisely.

Question: {input}

Previous interactions:
{agent_scratchpad}

Answer:
"""
)



# print(ollama_prompt.format(query="What is machine learning?"))


In [37]:
# quey from multiple tools
# Agent 
from langchain.agents import create_openai_tools_agent
agent = create_openai_tools_agent(llm,tools,prompt=ollama_prompt)

In [38]:
# agent exceuter
from langchain.agents.agent import AgentExecutor

agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)
agent_executor

AgentExecutor(verbose=True, agent=RunnableMultiActionAgent(runnable=RunnableAssign(mapper={
  agent_scratchpad: RunnableLambda(lambda x: format_to_openai_tool_messages(x['intermediate_steps']))
})
| PromptTemplate(input_variables=['agent_scratchpad', 'input'], input_types={}, partial_variables={}, template='\nYou are an AI assistant using the Gemma 2B model. Answer the following question concisely.\n\nQuestion: {input}\n\nPrevious interactions:\n{agent_scratchpad}\n\nAnswer:\n')
| RunnableBinding(bound=ChatOllama(model='gemma2:2b'), kwargs={'tools': [{'type': 'function', 'function': {'name': 'wikipedia', 'description': 'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.', 'parameters': {'properties': {'query': {'description': 'query to look up on wikipedia', 'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'funct

In [39]:
agent_executor.invoke({"input":"Tell me about the Langsmith"})



> Entering new AgentExecutor chain...
Langsmith is a powerful tool for crafting high-quality, customizable text prompts with specific instructions and guidance.  It allows you to create prompts that leverage AI's strengths to generate creative, informative, or engaging content in various formats like stories, poems, articles, emails, etc. 

Think of it as an "AI writing assistant" that helps users refine their ideas and get more out of text-based interactions with AI models.  


> Finished chain.


{'input': 'Tell me about the Langsmith',
 'output': 'Langsmith is a powerful tool for crafting high-quality, customizable text prompts with specific instructions and guidance.  It allows you to create prompts that leverage AI\'s strengths to generate creative, informative, or engaging content in various formats like stories, poems, articles, emails, etc. \n\nThink of it as an "AI writing assistant" that helps users refine their ideas and get more out of text-based interactions with AI models.  \n'}

In [40]:
agent_executor.invoke({"input":"What is machine learning?"})



> Entering new AgentExecutor chain...
Machine learning is a subfield of artificial intelligence (AI) that allows computers to learn from data without explicit programming.  They identify patterns and make predictions based on this data, improving their accuracy over time. 


> Finished chain.


{'input': 'What is machine learning?',
 'output': 'Machine learning is a subfield of artificial intelligence (AI) that allows computers to learn from data without explicit programming.  They identify patterns and make predictions based on this data, improving their accuracy over time. \n'}